# 08 — A second factor, and the product that prices correlation

## Why one factor is not enough

In a one-factor model every rate is a function of one number. Over a short interval the 2-year and the 30-year
rate move together with correlation exactly 1. Two things follow:

- one mean reversion cannot match swaption vols across both expiry and tenor;
- anything that depends on rates of different maturities moving apart is priced with a correlation nobody chose.

G2++ has two Gaussian factors with different mean reversions $\kappa_1, \kappa_2$, volatilities $\sigma_1, \sigma_2$ and
correlation $\rho$. The forward rate with time to maturity $\tau$ loads on them as

$$df(t, t+\tau) = \sigma_1 e^{-\kappa_1 \tau}\, dW_1 + \sigma_2 e^{-\kappa_2 \tau}\, dW_2 + \dots, \qquad d\langle W_1, W_2\rangle = \rho\, dt,$$

so the fast factor matters for short maturities and has died out for long ones. That is what lets the curve steepen
and flatten as well as shift.

This notebook reads the tables behind `docs/RESULTS.md` section 9. The first cell runs the C++ analysis (about 20
seconds the first time; cached afterwards).

In [ ]:
import _util
from _util import run_analysis
import numpy as np, pandas as pd, matplotlib.pyplot as plt

t = run_analysis("twofactor")
sorted(t)

## Calibrating the shape

Both models are fitted to 30 at-the-money swaptions. Each gets one volatility level per expiry row, so what is left
to explain is the tenor direction. The two-factor fit is started from four different points.

In [ ]:
par = t["g2pp_params"]
par[["model", "start", "kappa1", "kappa2", "sigma_ratio", "rho", "rms_bp", "max_bp", "status"]]

In [ ]:
fit = t["g2pp_fit"].copy()
fit["expiry"], fit["tenor"] = fit["expiry"].round().astype(int), fit["tenor"].round().astype(int)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col, title in zip(axes, ["error_1f_bp", "error_2f_bp"], ["one factor", "two factors"]):
    grid = fit.pivot(index="expiry", columns="tenor", values=col)
    im = ax.imshow(grid.values, cmap="RdBu_r", vmin=-4.5, vmax=4.5, aspect="auto")
    ax.set_xticks(range(grid.shape[1]), [f"{c}Y" for c in grid.columns]); ax.set_yticks(range(grid.shape[0]), [f"{r}Y" for r in grid.index])
    ax.set_xlabel("tenor"); ax.set_title(f"fit error, {title} (bp of normal vol)")
    for (i, j), v in np.ndenumerate(grid.values): ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=8)
axes[0].set_ylabel("expiry"); fig.colorbar(im, ax=axes, shrink=0.8); plt.show()

Things to be able to explain:

- The one-factor errors are a **pattern** across tenors, not noise: one mean reversion tilts the tenor direction one
  way and cannot do more.
- Three of the four starts reach the same two-factor answer; one stops in a one-factor-like basin (look at its
  $\rho$ and its RMS). Levenberg–Marquardt is a local method, so several starts are part of the calibration.
- The vol screen was generated from a two-factor model with $a = 40\%$, $b = 3\%$, $\rho = -0.70$
  (`tools/gen_vol_quotes.py`). Recovering it is a test of the calibration, not a statement about a market.

## A hand check: the correlation the fit implies

From the loadings above, the instantaneous correlation between forward rates with times to maturity $\tau_1$ and
$\tau_2$ is

$$\mathrm{corr} = \frac{a_1 a_2 + \rho\,(a_1 b_2 + a_2 b_1) + b_1 b_2}{\sqrt{a_1^2 + 2\rho a_1 b_1 + b_1^2}\,\sqrt{a_2^2 + 2\rho a_2 b_2 + b_2^2}},
\qquad a_i = \sigma_1 e^{-\kappa_1 \tau_i},\; b_i = \sigma_2 e^{-\kappa_2 \tau_i}.$$

Computed here in numpy from the fitted parameters, it must agree with the `corr_1y_10y` column the C++ wrote.

In [ ]:
best = par[par.model == "2F"].sort_values("rms_bp").iloc[0]
k1, k2, ratio, rho = best.kappa1, best.kappa2, best.sigma_ratio, best.rho

def fwd_corr(t1, t2):
    a1, b1, a2, b2 = np.exp(-k1 * t1), ratio * np.exp(-k2 * t1), np.exp(-k1 * t2), ratio * np.exp(-k2 * t2)
    return (a1 * a2 + rho * (a1 * b2 + a2 * b1) + b1 * b2) / np.sqrt((a1**2 + 2 * rho * a1 * b1 + b1**2) * (a2**2 + 2 * rho * a2 * b2 + b2**2))

print(f"by hand: corr(1Y, 10Y) = {fwd_corr(1.0, 10.0):.4f}    C++: {best.corr_1y_10y:.4f}")
tau = np.linspace(0.25, 30, 200)
for anchor in (1.0, 5.0, 10.0): plt.plot(tau, fwd_corr(anchor, tau), label=f"against the {anchor:.0f}Y forward")
plt.axhline(1.0, color="grey", ls="--", lw=1); plt.text(20, 1.005, "one factor", color="grey", fontsize=8)
plt.xlabel("time to maturity of the other forward rate, years"); plt.ylabel("instantaneous correlation"); plt.legend(); plt.show()

## The Bermudan: same Europeans, different correlation

Holding the shape, the volatility level is bootstrapped to the nine co-terminal swaptions of the 10-year swap
callable from year 1. Every row below reprices all nine (`strip_error_bp`). Only the correlation between rates on
different exercise dates differs.

In [ ]:
ber = t["g2pp_bermudan"]
scan = ber[ber.case == "fast second factor"]
plt.errorbar(scan.rho, scan.price_bp, yerr=2 * scan.se_bp, fmt="o-", capsize=3, label="two factors (Monte Carlo lower bound)")
plt.axhline(ber.price_bp[0], color="grey", ls="--", label="one factor (PDE)")
plt.axhline(ber.best_european_bp[0], color="lightgrey", label="best European")
plt.xlabel("correlation between the two factors"); plt.ylabel("bp of notional"); plt.legend(); plt.show()
ber[["case", "rho", "strip_error_bp", "price_bp", "se_bp", "method"]]

Two checks worth saying out loud:

- With the second factor switched off, the two-factor Monte Carlo must give the one-factor PDE price. It does, to
  within its standard error (rows 0 and 1).
- The price moves by tens of basis points with $\rho$ while every European stays repriced. European swaptions fix the
  variance of each swap rate on its own date and say nothing about how rates on different dates move together.

## A CMS spread option

The option pays $\max(S_{10} - S_2 - K, 0)$ in five years. The variance of the spread is

$$\sigma_{spread}^2 = \sigma_{10}^2 + \sigma_2^2 - 2\,\rho_{rates}\,\sigma_{10}\,\sigma_2 .$$

The two vols are swaption vols. The correlation between the two rates is in no swaption price. Every model in the
table reprices both swaptions exactly, so they differ only in $\rho_{rates}$.

In [ ]:
cms, info = t["cms_spread"], t["cms_spread_info"].iloc[0]
ok = cms[cms.attainable == 1].copy()
print("no model reprices both swaptions at rho =", list(cms[cms.attainable == 0].rho))
ok[["model", "rho", "sigma_ratio", "rate_correlation", "spread_vol_bp", "forward_spread_bp", "price_bp"]]

### Two hand checks

1. **The spread's volatility from the formula.** Using the market vols of the two swaptions and the model's rate
   correlation, the formula above should give the model's spread vol (not exactly: the rates' volatilities at expiry
   differ slightly from the swaption vols by convexity).
2. **The price from Bachelier.** If the spread at expiry is roughly normal with that volatility, the option is
   $D(T)\,[(F - K)\,\Phi(d) + \sigma\sqrt{T}\,\varphi(d)]$ with $d = (F - K)/(\sigma\sqrt{T})$, where $F$ is the model's
   expected spread.

In [ ]:
from math import erf, exp, pi, sqrt
Phi = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
phi = lambda x: exp(-0.5 * x * x) / sqrt(2 * pi)
T, df, K = info.expiry_years, info.df_expiry, info.curve_spread_bp
sL, sS = info.long_vol_bp, info.short_vol_bp

rows = []
for r in ok.itertuples():
    vol_formula = sqrt(max(sL**2 + sS**2 - 2 * r.rate_correlation * sL * sS, 0.0))
    sd = r.spread_vol_bp * sqrt(T)
    d = (r.forward_spread_bp - K) / sd
    bachelier = df * ((r.forward_spread_bp - K) * Phi(d) + sd * phi(d))
    rows.append((r.model, r.rho, r.spread_vol_bp, vol_formula, r.price_bp, bachelier))
pd.DataFrame(rows, columns=["model", "rho", "spread vol, model", "spread vol, formula", "price, model", "price, Bachelier"]).round(2)

What to take from the table:

- **One factor prices the option at its intrinsic value.** With both vols matched the rates are perfectly correlated,
  the spread has almost no volatility, and what is left is the difference in convexity between the two rates.
- **Under two factors the price is a reading of the correlation.** That is why spread options are what a desk marks
  $\rho$ to.
- **The two swaption vols already bound $\rho$**: for the values listed as not attainable, no volatility ratio
  reprices both. With these speeds, a 10-year vol as high as the 2-year one needs strongly negative correlation.

Simplification to mention if asked: this is one payment at expiry with unit accrual, not a traded strip of caplets.